In [67]:
# importing libraries
import numpy as np                  # for numerical operations
import pandas as pd                 # for data manipulation andanalysis
import matplotlib.pyplot as plt     # for data visualization
%matplotlib inline 
from wordcloud import WordCloud     # WordCloud for text visualization
import nltk                         # nltk for natural lang processing
from nltk.corpus import stopwords   # for stopwords

# dowloading nltk data
nltk.download('stopwords')          # downloading stopwords data, used for removing stopwords like the, is, at, on, etc
nltk.download('punkt')              # downloading tokenizer data, used to tokenize
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abakare\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\abakare\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abakare\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [68]:
df = pd.read_csv('spam.csv')           # read the CSV file
df.head()                              # display the first few rows of dataframe

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [69]:
df.drop(columns = ['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True) 
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [70]:
# rename the column names
df.rename(columns={'v1':'target','v2':'text'}, inplace=True)
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Data Processing

In [71]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
# converting text labels of col 'target' into nums
df['target'] = encoder.fit_transform(df['target'])     

df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [72]:
df.duplicated().sum()       # check duplicate value

np.int64(403)

In [73]:
len(df)

5572

In [74]:
df = df.drop_duplicates(keep='first')       # keep the first row & delete all subsequent duplicates
len(df)

5169

Feature engineering

In [75]:
# importing Porter Stemmer for text stemming
from nltk.stem.porter import PorterStemmer
import string

ps = PorterStemmer()

In [76]:
# Lowercase transformation and text preprocessing function
def transform_text(text):
    # transform to lowercase
    text = text.lower()

    # tokenization using nltk
    text = nltk.word_tokenize(text)

    # removing special chars
    y = []
    for i in text:
        if i.isalnum():
            y.append(i)
    # removing special characters
    text = y[:]         # copies y list into text
    y.clear()

    stop_words = set(stopwords.words('english'))
    # loop through the tokens and remove stopwords and punctuations
    for i in text:
        if i not in stop_words and i not in string.punctuation:
            y.append(i)
    # Stemming using Porter Stemmer means "loving", "loved", and "loves" all become "love"
    text = y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))
    return " ".join(y)      # join the tokens back into single string

In [77]:
transform_text('Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...')

'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [78]:
df['transformed_text'] = df['text'].apply(transform_text)
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [79]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
tfid = TfidfVectorizer(max_features=500)
# TF-IDF(Term Frequency-Inverse Document Frequency tool) 
# It evaluates how important a word is to a document in your dataset. If a word appears often in one specific text but rarely in other texts (like "vaccine" in a medical article), it gets a high score. If a word appears everywhere (like "the" or "is"), it gets a low score.
# We are telling the tool to only keep the top 500 most important words across dataset

In [80]:
x = tfid.fit_transform(df['transformed_text']).toarray()
# fit_transform: The vectorizer first learns the vocabulary of your text (fit) and then converts the text into TF-IDF scores (transform)
# .toarray() converts it into a standard 2D NumPy array,so you can easily feed it into a model.
y = df['target'].values
# Result (y): Your dependent variable (Target/Label). This is a 1D array containing what you want to predict (e.g., 1 for Spam and 0 for Ham, or 1 for Positive sentiment and 0 for Negative sentiment).

Model training

In [81]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [82]:
svc = SVC(kernel= "sigmoid", gamma  = 1.0)
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth = 5)
lrc = LogisticRegression(solver = 'liblinear', penalty = 'l1')
rfc = RandomForestClassifier(n_estimators = 50, random_state = 2 )
abc = AdaBoostClassifier(n_estimators = 50, random_state = 2)
bc = BaggingClassifier(n_estimators = 50, random_state = 2)
etc = ExtraTreesClassifier(n_estimators = 50, random_state = 2)
gbdt = GradientBoostingClassifier(n_estimators = 50, random_state = 2)    
xgb  = XGBClassifier(n_estimators = 50, random_state = 2)

In [93]:
clfs = {
    'SVC': svc,
    'KNN': knc,
    'NB': mnb,
    'DT': dtc,
    'LR': lrc,
    'RF': rfc,
    'Adaboost': abc,
    'Bgc': bc,
    'ETC': etc,
    'GBDT': gbdt,
    'xgb': xgb
    
}

Model Evaluation

In [94]:
from sklearn.metrics import accuracy_score, precision_score
def train_classifier(clf, X_train, y_train, X_test, y_test):
    clf.fit(X_train,y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    return accuracy , precision

In [99]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)
accuracy_scores = []
precision_scores = []
for name , clf in clfs.items():
    current_accuracy, current_precision = train_classifier(clf, X_train, y_train, X_test, y_test)
    print()
    print("For: ", name)
    print("Accuracy: ", current_accuracy)
    print("Precision: ", current_precision)
    
    accuracy_scores.append(current_accuracy)
    precision_scores.append(current_precision)


For:  SVC
Accuracy:  0.9680851063829787
Precision:  0.9487179487179487

For:  KNN
Accuracy:  0.9274661508704062
Precision:  1.0

For:  NB
Accuracy:  0.9709864603481625
Precision:  0.9655172413793104

For:  DT
Accuracy:  0.9381044487427466
Precision:  0.8854166666666666

For:  LR
Accuracy:  0.9622823984526112
Precision:  0.9541284403669725


c:\Projects\MLops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Projects\MLops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



For:  RF
Accuracy:  0.9729206963249516
Precision:  0.9435483870967742

For:  Adaboost
Accuracy:  0.9235976789168279
Precision:  0.8734177215189873

For:  Bgc
Accuracy:  0.9593810444874274
Precision:  0.8870967741935484

For:  ETC
Accuracy:  0.971953578336557
Precision:  0.9291338582677166

For:  GBDT
Accuracy:  0.9526112185686654
Precision:  0.9405940594059405

For:  xgb
Accuracy:  0.9671179883945842
Precision:  0.956140350877193
